# Data Setup — Agricultural Extension RAG (Kaggle)

This notebook shows how to **download, load, and sanity-check** the
competition dataset, and reproduces the provided TF-IDF baseline
(nDCG@5 ≈ 0.551) as a smoke test.

**Source:** https://www.kaggle.com/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/data

Run top to bottom. If you're on Kaggle itself, skip the download cell and
just attach the competition as a data source via the Input panel instead.


## 1. Download the data

Two ways to get the data locally (skip this whole section if running
inside a Kaggle Notebook with the competition attached as an input):

**A. Kaggle CLI** (needs a Kaggle account + API token, and you must have
joined the competition on its page first):

```bash
pip install kaggle --break-system-packages
mkdir -p ~/.kaggle && mv ~/Downloads/kaggle.json ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json
kaggle competitions download -c agricultural-extension-rag-smart-retrieval-for-farmers -p data/
unzip -o data/agricultural-extension-rag-smart-retrieval-for-farmers.zip -d data/
```

**B. Kaggle web UI:** open the [Data
tab](https://www.kaggle.com/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/data),
click **Join Competition**, then **Download All**, and unzip into `data/`.

The cell below tries a few likely locations for the CSVs so the notebook
works whether you're on Kaggle or running locally with a `data/` folder.


In [ ]:
# ==========================================================================
# 1. Locate + load the data
# ==========================================================================
import pandas as pd
from pathlib import Path

CANDIDATE_DIRS = [
    Path('/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers'),
    Path('data'),
    Path('.'),
]

data_dir = None
for d in CANDIDATE_DIRS:
    if (d / 'documents.csv').exists():
        data_dir = d
        break

if data_dir is None:
    # Fall back to searching more broadly (e.g. a nested Kaggle input mount).
    hits = list(Path('/kaggle/input').glob('**/documents.csv')) if Path('/kaggle/input').exists() else []
    if hits:
        data_dir = hits[0].parent
    else:
        raise FileNotFoundError(
            "documents.csv not found. Download the data first (see Section 1 above) "
            "into a local 'data/' folder, or attach the competition as a Kaggle input."
        )

print('Reading data from:', data_dir)

documents      = pd.read_csv(data_dir / 'documents.csv')
train_queries  = pd.read_csv(data_dir / 'train_queries.csv')
qrels          = pd.read_csv(data_dir / 'qrels_train.csv')
test_queries   = pd.read_csv(data_dir / 'test_queries.csv')

# sample_submission / baseline_submission are optional but useful for format checks
sample_submission_path = data_dir / 'sample_submission.csv'
sample_submission = pd.read_csv(sample_submission_path) if sample_submission_path.exists() else None

print('documents:', documents.shape)
print('train_queries:', train_queries.shape)
print('qrels_train:', qrels.shape)
print('test_queries:', test_queries.shape)

## 2. Sanity-check the schema and a few rows

Quick checks that the expected columns are present and a peek at the data,
so you know what you're working with before writing any modeling code.


In [ ]:
# ==========================================================================
# 2. Schema + sample rows
# ==========================================================================
expected_cols = {
    'documents.csv': {'document_id','title','text','source','crop','country','origin','source_url','license'},
    'train_queries.csv': {'query_id','query','positive_docs'},
    'qrels_train.csv': {'query_id','document_id','relevance'},
    'test_queries.csv': {'query_id','query'},
}
frames = {'documents.csv': documents, 'train_queries.csv': train_queries,
          'qrels_train.csv': qrels, 'test_queries.csv': test_queries}

for name, cols in expected_cols.items():
    missing = cols - set(frames[name].columns)
    status = 'OK' if not missing else f'MISSING: {missing}'
    print(f'{name:20s} -> {status}')

print()
display(documents.head(3))
display(train_queries.head(3))
display(qrels.head(3))
display(test_queries.head(3))

## 3. Relevance label distribution

`qrels_train.csv` uses a graded 0–3 relevance scale (3 = perfect match,
0 = not relevant, including deliberately tricky hard negatives). Worth
checking the label balance before modeling.


In [ ]:
# ==========================================================================
# 3. Label distribution
# ==========================================================================
print(qrels['relevance'].value_counts().sort_index())
print()
print('Docs per training query (avg):', qrels.groupby('query_id').size().mean().round(2))
print('Positive (relevance>=1) docs per query (avg):',
      qrels[qrels['relevance']>=1].groupby('query_id').size().mean().round(2))

## 4. Baseline: TF-IDF cosine similarity

Reproduces the kind of retriever behind `baseline_submission.csv`
(nDCG@5 = 0.551 on the hidden test set) — a minimal, dependency-light
starting point and a good smoke test that the data is loaded correctly.


In [ ]:
# ==========================================================================
# 4. TF-IDF baseline retriever
# ==========================================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Represent each document as its title + body text.
doc_text = documents['title'].fillna('') + '. ' + documents['text'].fillna('')

vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2)
doc_mat = vec.fit_transform(doc_text)

def rank_documents(queries_series, k=5):
    """Return {query_id: [doc_id, ...]} of the top-k TF-IDF matches."""
    sims = cosine_similarity(vec.transform(queries_series), doc_mat)
    out = {}
    return sims

sims_train = cosine_similarity(vec.transform(train_queries['query']), doc_mat)
train_pred = {}
for i, qid in enumerate(train_queries['query_id']):
    top_idx = sims_train[i].argsort()[::-1][:5]
    train_pred[qid] = documents.iloc[top_idx]['document_id'].tolist()

print('Built TF-IDF predictions for', len(train_pred), 'training queries.')

## 5. Local evaluation: nDCG@5

Implements the competition's own metric so the baseline (or any model) can
be validated **locally against the training labels**, without needing a
leaderboard submission. Never run this against hidden test labels — you
don't have them; this is for `train_queries.csv` / `qrels_train.csv` only.


In [ ]:
# ==========================================================================
# 5. nDCG@5 metric + baseline score
# ==========================================================================
import numpy as np

def dcg(rels, k=5):
    v = np.asarray(list(rels)[:k], dtype=float)
    if len(v) == 0: return 0.0
    return float(np.sum(v / np.log2(np.arange(2, len(v) + 2))))

def evaluate_ndcg_at_5(predictions, qrels_frame=qrels):
    """predictions: {query_id: [doc_id, ...]} already ranked best-first."""
    lookup = {qid: dict(zip(g['document_id'], g['relevance']))
              for qid, g in qrels_frame.groupby('query_id')}
    scores = {}
    for qid, judged in lookup.items():
        ranked = predictions.get(qid, [])[:5]
        gains  = [judged.get(d, 0) for d in ranked]
        ideal  = sorted(judged.values(), reverse=True)[:5]
        idcg   = dcg(ideal)
        scores[qid] = dcg(gains) / idcg if idcg else 0.0
    return float(np.mean(list(scores.values()))), scores

baseline_score, _ = evaluate_ndcg_at_5(train_pred)
print(f'TF-IDF baseline nDCG@5 on TRAIN queries: {baseline_score:.4f}')
print('(Reference: the official baseline scores nDCG@5 = 0.551 on the hidden TEST set.)')

## 6. Build a submission file

Generates predictions for `test_queries.csv` in the exact long format the
competition requires, then validates the file's shape before writing it.


In [ ]:
# ==========================================================================
# 6. Build + validate submission.csv
# ==========================================================================
sims_test = cosine_similarity(vec.transform(test_queries['query']), doc_mat)

rows = []
for i, qid in enumerate(test_queries['query_id']):
    top_idx = sims_test[i].argsort()[::-1][:5]      # best-first order
    for idx in top_idx:
        rows.append({'QueryId': str(qid), 'DocumentId': str(documents.iloc[idx]['document_id'])})

submission = pd.DataFrame(rows, columns=['QueryId', 'DocumentId'])

# --- Format validation, matching sample_submission.csv's structure --------
assert list(submission.columns) == ['QueryId', 'DocumentId']
assert len(submission) == len(test_queries) * 5, len(submission)
assert submission.groupby('QueryId', sort=False).size().eq(5).all()
assert set(submission['QueryId']) == set(test_queries['query_id'].astype(str))
assert not submission.duplicated(['QueryId', 'DocumentId']).any()

submission.to_csv('submission.csv', index=False)
print('Saved submission.csv:', submission.shape)
submission.head(10)

## Next steps

This notebook only reproduces the TF-IDF baseline (nDCG@5 ≈ 0.55). For a
stronger retriever, see the full pipeline notebook in this repo, which adds:

- a rule-based **slot extractor** (crop / issue / intent / zone) for
  intent-aware matching beyond bag-of-words overlap,
- **BM25** candidate generation,
- a **LightGBM LambdaMART** reranker on engineered features,
- a fine-tuned **cross-encoder** (`BAAI/bge-reranker-base`) second-stage
  reranker,
- **leak-free `GroupKFold` cross-validation** at every stage before
  trusting a score.
